# 🚆 สถิติผู้โดยสารระบบขนส่งสาธารณะในประเทศไทย
**วิเคราะห์ปริมาณการเดินทางของประชาชนด้วยระบบขนส่งสาธารณะ**

**ชุดข้อมูล:** สถิติผู้โดยสารรายวันของระบบขนส่งสาธารณะทั่วประเทศไทย (ปี 2568–2569)  
**ช่วงเวลา:** ประมาณ 14 เดือน  
**แหล่งข้อมูล:** กระทรวงคมนาคม

---

## วัตถุประสงค์การวิเคราะห์
1. **สัดส่วนการใช้ระบบขนส่ง (Modal Share)** — ระบบใดมีผู้โดยสารมากที่สุด?
2. **การเปรียบเทียบรถไฟฟ้าในเมือง (Urban Rail Comparison)** — พฤติกรรมผู้โดยสารแต่ละสายแตกต่างกันอย่างไร?
3. **การตรวจจับเหตุการณ์พิเศษ (Event Detection)** — วันหยุดและเทศกาลปรากฏในข้อมูลหรือไม่?
4. **การพยากรณ์ (Forecasting)** — คาดการณ์ความต้องการผู้โดยสาร 30 วันข้างหน้าด้วย Facebook Prophet

In [ ]:
# ติดตั้งแพ็กเกจที่จำเป็น (รันครั้งเดียวใน Colab)
!pip install prophet plotly scikit-learn -q

In [1]:
# นำเข้าไลบรารีทั้งหมดที่ใช้ในการวิเคราะห์
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
import warnings
warnings.filterwarnings('ignore')

print('✅ นำเข้าไลบรารีสำเร็จ')

/Users/sarat/Code/superai-se6/superai-se6-mini-hackathon-1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ นำเข้าไลบรารีสำเร็จ


---
## โหลดข้อมูล (Data Loading)

### เป้าหมาย
โหลดชุดข้อมูลและตรวจสอบโครงสร้างเบื้องต้น

**ชุดข้อมูลที่ใช้:**
- `passengers68.csv` — ข้อมูลผู้โดยสารปี 2568
- `passengers69.csv` — ข้อมูลผู้โดยสารปี 2569

In [4]:
# =========================
# 1) โหลดข้อมูล
# =========================
df68 = pd.read_csv(
    "https://raw.githubusercontent.com/lumduan/superai-se6-mini-hackathon-1/main/dataset-5/data/passengers68.csv"
)

df69 = pd.read_csv(
    "https://raw.githubusercontent.com/lumduan/superai-se6-mini-hackathon-1/main/dataset-5/data/passengers69.csv"
)


# =========================
# 2) ตรวจสอบจำนวนคอลัมน์
# =========================
print("=== Column Count ===")
print(f"df68 columns : {df68.shape[1]}")
print(f"df69 columns : {df69.shape[1]}")


# =========================
# 3) ตรวจสอบชื่อคอลัมน์
# =========================
cols68 = set(df68.columns)
cols69 = set(df69.columns)

print("\n=== Column Name Check ===")

if cols68 == cols69:
    print("Column names match")
else:
    print("Column names DO NOT match")

    only_68 = cols68 - cols69
    only_69 = cols69 - cols68

    if only_68:
        print("Columns only in df68:")
        print(sorted(only_68))

    if only_69:
        print("Columns only in df69:")
        print(sorted(only_69))


# =========================
# 4) เรียง column ให้เหมือนกัน (กัน order ต่าง)
# =========================
df69 = df69[df68.columns]


# =========================
# 5) รวมข้อมูล
# =========================
df = pd.concat([df68, df69], ignore_index=True)


# =========================
# 6) สรุปผล
# =========================
print("\n=== Dataset Summary ===")
print(f"df68 จำนวนแถว: {df68.shape[0]:,}  คอลัมน์: {df68.shape[1]}")
print(f"df69 จำนวนแถว: {df69.shape[0]:,}  คอลัมน์: {df69.shape[1]}")
print(f"รวมทั้งหมด:   {df.shape[0]:,}  คอลัมน์: {df.shape[1]}")

df.head()

=== Column Count ===
df68 columns : 8
df69 columns : 8

=== Column Name Check ===
Column names match

=== Dataset Summary ===
df68 จำนวนแถว: 69,440  คอลัมน์: 8
df69 จำนวนแถว: 3,010  คอลัมน์: 8
รวมทั้งหมด:   72,450  คอลัมน์: 8


,รูปแบบการเดินทาง,วัตถุประสงค์,สาธารณะ/ส่วนบุคคล,หน่วยงาน,ยานพาหนะ/ท่า,วันที่,หน่วย,ปริมาณ
0,ทางถนน,การเดินทางระหว่างจังหวัด,สาธารณะ,บขส.,รถ บขส. และ รถร่วม,01/01/2025,คน,"127,551"
1,ทางถนน,การเดินทางระหว่างจังหวัด,สาธารณะ,ขบ.,รถหมวด 3,01/01/2025,คน,"8,218"
2,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,ทล.,รถยนต์เฉพาะ 4 ล้อ (10 จุดสำรวจ),01/01/2025,คัน,"877,943"
3,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,ทล.,รถยนต์ทุกประเภท (10 จุดสำรวจ),01/01/2025,คัน,"932,642"
4,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,กทพ.,รถยนต์เฉพาะ 4 ล้อ (ทางด่วน),01/01/2025,คัน,"1,364,992"


In [5]:
print("=== ข้อมูลโครงสร้าง DataFrame ===")

display(pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.values
}))

=== ข้อมูลโครงสร้าง DataFrame ===


,column,dtype
0,รูปแบบการเดินทาง,str
1,วัตถุประสงค์,str
2,สาธารณะ/ส่วนบุคคล,str
3,หน่วยงาน,str
4,ยานพาหนะ/ท่า,str
5,วันที่,str
6,หน่วย,str
7,ปริมาณ,str


---
## ทำความสะอาดและตรวจสอบข้อมูล (Data Cleaning & Validation)

In [6]:
# =====================================================
# แปลงคอลัมน์วันที่ + ตรวจสอบช่วงข้อมูล
# =====================================================

print("🗓️ Date Processing")

# แปลงวันที่ (ค่าที่ผิด format จะกลายเป็น NaT)
df["date"] = pd.to_datetime(df["วันที่"], dayfirst=True, errors="coerce")

# ตรวจ missing date
missing_date_rows = df["date"].isna().sum()
print(f"แถวที่แปลงวันที่ไม่ได้: {missing_date_rows:,}")

# ตรวจ year anomaly
df["year"] = df["date"].dt.year

year_counts = df["year"].value_counts().sort_index()

print("\n📅 Distribution ของปีใน dataset")
display(year_counts)

# ตรวจ year ที่น่าจะผิดปกติ
year_anomaly = df[(df["year"] < 2015) | (df["year"] > 2030)]

if len(year_anomaly) > 0:
    print("\n⚠️ พบปีที่อาจผิดพลาด:")
    display(year_anomaly.head())

# เรียงข้อมูลตามเวลา
df = df.sort_values("date").reset_index(drop=True)

print(f"\nช่วงวันที่: {df['date'].min().date()} → {df['date'].max().date()}")

total_days = (df["date"].max() - df["date"].min()).days + 1
print(f"จำนวนวันทั้งหมดในช่วงเวลา: {total_days:,} วัน")

# =====================================================
# ตรวจ date gap
# =====================================================

print("\n📉 ตรวจสอบวันที่ขาดหาย (Date Gap)")

all_dates = pd.date_range(df["date"].min(), df["date"].max(), freq="D")

missing_dates = all_dates.difference(df["date"])

print(f"จำนวนวันที่ขาดหาย: {len(missing_dates):,}")

if len(missing_dates) > 0:
    display(missing_dates[:10])

🗓️ Date Processing
แถวที่แปลงวันที่ไม่ได้: 53,744

📅 Distribution ของปีใน dataset


year
2025.0    15696
2026.0     3010
Name: count, dtype: int64


ช่วงวันที่: 2025-01-01 → 2026-03-11
จำนวนวันทั้งหมดในช่วงเวลา: 435 วัน

📉 ตรวจสอบวันที่ขาดหาย (Date Gap)
จำนวนวันที่ขาดหาย: 0


In [ ]:
print("🔎 ตรวจสอบแถวที่ว่างทั้งแถว")
# นับแถวที่ว่างทั้งหมด
all_nan_mask = df.isna().all(axis=1)
all_nan_rows = all_nan_mask.sum()

print(f"\nจำนวนแถวที่ว่างทั้งหมด: {all_nan_rows:,}")



🔎 ตรวจสอบแถวที่ว่างทั้งแถว

จำนวนแถวที่ว่างทั้งหมด: 0

✅ ไม่พบแถวที่ว่างทั้งหมด ไม่ต้องลบ
จำนวนแถวปัจจุบัน: 18,706 แถว


,รูปแบบการเดินทาง,วัตถุประสงค์,สาธารณะ/ส่วนบุคคล,หน่วยงาน,ยานพาหนะ/ท่า,วันที่,หน่วย,ปริมาณ,date,year
0,ทางถนน,การเดินทางระหว่างจังหวัด,สาธารณะ,บขส.,รถ บขส. และ รถร่วม,01/01/2025,คน,"127,551",2025-01-01,2025.0
1,ทางอากาศ,การเดินทางระหว่างจังหวัด,สาธารณะ,ทอท.,ท่าอากาศยานดอนเมือง,01/01/2025,คน,"31,842",2025-01-01,2025.0
2,ทางอากาศ,การเดินทางระหว่างจังหวัด,สาธารณะ,ทอท.,ท่าอากาศอื่น ๆ ของ ทอท.,01/01/2025,คน,"28,425",2025-01-01,2025.0
3,ทางอากาศ,การเดินทางระหว่างจังหวัด,สาธารณะ,ทย.,ท่าอากาศยานภูมิภาค,01/01/2025,คน,"20,579",2025-01-01,2025.0
4,ทางอากาศ,การเดินทางระหว่างจังหวัด,สาธารณะ,กพท.,ท่าอากาศยานอู่ตะเภา,01/01/2025,คน,137,2025-01-01,2025.0


In [6]:
print("🔎 ตรวจสอบ Missing Values")

missing_df = (
    df.isna()
    .sum()
    .reset_index()
)

missing_df.columns = ["column", "missing_values"]

display(missing_df)

🔎 ตรวจสอบ Missing Values


,column,missing_values
0,รูปแบบการเดินทาง,53744
1,วัตถุประสงค์,53744
2,สาธารณะ/ส่วนบุคคล,53744
3,หน่วยงาน,53744
4,ยานพาหนะ/ท่า,53744
5,วันที่,53744
6,หน่วย,53744
7,ปริมาณ,54188
8,date,53744
9,year,53744
